# Australia's National Greenhouse Gas inventory

In [1]:
"""
Charts from the Australian National Greenhouse Gas Inventory.

Note: you will need to download the latest XLS file to
the CACHE/ANGG_greenhouse directory.
"""

'\nCharts from the Australian National Greenhouse Gas Inventory.\n\nNote: you will need to download the latest XLS file to\nthe CACHE/ANGG_greenhouse directory.\n'

## Set-up

In [2]:
from pandas import DataFrame, PeriodIndex, Series, read_excel
from mgplot import seastrend_plot_finalise, line_plot_finalise
from abs_helper import set_chart_dir, clear_chart_dir

from abs_population import get_population

In [3]:
CHART_DIR = "./CHARTS/ANGG/"
set_chart_dir(CHART_DIR)
clear_chart_dir()
SHOW = False

## Data Acquisition

In [4]:
# Million tonnes of carbon dioxide equivalent (Mt CO2-e)
ANGG_FILE = "./CACHE/ANGG_greenhouse/nggi-quarterly-update-december-2024.xlsx"


def get_angg() -> DataFrame:
    """Quarterly national greenhouse gas inventory, from the cached workbook."""
    frame = read_excel(
        ANGG_FILE, sheet_name="Figure 1", index_col=0, skiprows=5
    ).dropna(how="all", axis=0)
    frame.index = PeriodIndex(frame.index, freq="Q")
    return frame


def report_population(pop: Series, units: str) -> None:
    """Print the population units and the most recent quarters."""
    print(f"Population units: {units}, {pop.tail(6)}")


population, pop_units = get_population(state="Australia")
report_population(population, pop_units)
angg = get_angg()

Population units: Persons, Series ID
2025Q1    2.752978e+07
2025Q2    2.761103e+07
2025Q3    2.772238e+07
2025Q4    2.780102e+07
2026Q1    2.787988e+07
2026Q2    2.795897e+07
Freq: Q-DEC, Name: A2060842F, dtype: float64


## Plotting

In [5]:
TONNES_PER_MEGATONNE = 1_000_000

COMMON = {
    "legend": True,
    "show": SHOW,
}
EMISSIONS_TITLE = "Australia's Greenhouse Gas Emissions"
EMISSIONS_YLABEL = "Million tonnes $CO_{{2}}$-e / Quarter"


def plot_emissions_level() -> None:
    """Total quarterly emissions, original series."""
    line_plot_finalise(
        angg[angg.columns[0]],
        tag="lineplot",
        annotate=True,
        title=EMISSIONS_TITLE,
        ylabel=EMISSIONS_YLABEL,
        rfooter="NGGI",
        **COMMON,
    )


def plot_emissions_seastrend() -> None:
    """Quarterly emissions: seasonally adjusted against trend."""
    seastrend_plot_finalise(
        angg[angg.columns[1:]],
        tag="seastrend",
        title=EMISSIONS_TITLE,
        ylabel=EMISSIONS_YLABEL,
        rfooter="NGGI",
        **COMMON,
    )


def plot_emissions_per_capita() -> None:
    """Trend emissions per head of population, in tonnes per quarter."""
    per_capita = angg[angg.columns[2]] * TONNES_PER_MEGATONNE / population
    per_capita.name = "Trend per Capita"
    line_plot_finalise(
        per_capita,
        tag="lineplot",
        annotate=True,
        title=f"{EMISSIONS_TITLE} per Capita",
        ylabel="Tonnes $CO_{2}$-e / Quarter",
        rfooter="NGGI, ABS 3101.0",
        **COMMON,
    )


plot_emissions_level()
plot_emissions_seastrend()
plot_emissions_per_capita()

## Finished

In [6]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-08-30 18:04:43

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.16.1

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot: 0.2.33
pandas: 3.0.5

Watermark: 2.6.0

